# Week 72, Day 4: Coverage and mutation testing

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w072d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def grade(score):
    if score >= 70:
        return "distinction"
    if score >= 40:
        return "pass"
    return "fail"

def test_grade():
    grade(85)
    grade(50)
    grade(10)

test_grade()
print("every line of grade ran, and the test passed")

In [ ]:
import ast, sys

source = '''def grade(score):
    if score >= 70:
        return "distinction"
    if score >= 40:
        return "pass"
    return "fail"
'''
code = compile(source, "grading.py", "exec")
executable = {n.lineno for n in ast.walk(ast.parse(source)) if isinstance(n, ast.stmt)}
executed = set()

def tracer(frame, event, arg):
    if frame.f_code.co_filename != "grading.py":
        return None
    if event == "line":
        executed.add(frame.f_lineno)
    return tracer

ns = {}
sys.settrace(tracer)
try:
    exec(code, ns)                  # running the module defines grade (line 1)
    ns["grade"](85)
    ns["grade"](10)
finally:
    sys.settrace(None)

missing = sorted(executable - executed)
print(f"{len(executed & executable)} of {len(executable)} lines, {100 * len(executed & executable) // len(executable)}%")
print("missing:", missing)

In [ ]:
import sys

def clamp(x, low, high):
    if x < low:
        x = low
    if x > high:
        x = high
    return x

arcs = set()

def tracer(frame, event, arg):
    if frame.f_code is not clamp.__code__:
        return None
    last = [frame.f_lineno]
    def local(frame, event, arg):
        if event == "line":
            arcs.add((last[0], frame.f_lineno))
            last[0] = frame.f_lineno
        return local
    return local

start = clamp.__code__.co_firstlineno
sys.settrace(tracer)
try:
    clamp(-5, 0, 10)          # every line runs...
finally:
    sys.settrace(None)
print(sorted((a - start, b - start) for a, b in arcs))
print("never jumped from the first if straight to the second (x >= low):", (start + 1, start + 3) not in arcs)

In [ ]:
import ast

source = '''def grade(score):
    if score >= 70:
        return "distinction"
    if score >= 40:
        return "pass"
    return "fail"
'''
SWAPS = {ast.GtE: ast.Gt, ast.Gt: ast.GtE, ast.Lt: ast.LtE, ast.LtE: ast.Lt}

def mutants(source):
    tree = ast.parse(source)
    count = sum(1 for n in ast.walk(tree) if isinstance(n, ast.Compare))
    for i in range(count):
        tree = ast.parse(source)
        target = [n for n in ast.walk(tree) if isinstance(n, ast.Compare)][i]
        old = type(target.ops[0])
        target.ops[0] = SWAPS[old]()
        yield f"line {target.lineno}: {old.__name__} -> {SWAPS[old].__name__}", ast.unparse(tree)

def weak_tests(grade):
    assert grade(85) == "distinction" and grade(50) == "pass" and grade(10) == "fail"

def strong_tests(grade):
    weak_tests(grade)
    assert grade(70) == "distinction" and grade(69) == "pass" and grade(40) == "pass" and grade(39) == "fail"

for tests in (weak_tests, strong_tests):
    survived = []
    for label, mutated in mutants(source):
        ns = {}
        exec(mutated, ns)
        try:
            tests(ns["grade"])
            survived.append(label)
        except AssertionError:
            pass
    print(tests.__name__, "survivors:", survived)

> **Key idea.** sys.settrace installs a tracer that receives call, line, return and exception events; returning a local tracer from the call event traces that frame. Line coverage compares executed lines with the module's statement lines and reports the missing ones; branch coverage records arcs between lines and finds paths never taken. Coverage shows what ran, not what was checked, and can't see missing code or untested inputs: low coverage is a warning, high coverage weak evidence. Mutation testing plants small bugs with ast, runs the suite against each mutant, and reports survivors; the mutation score is killed over total, and equivalent mutants can't be killed.

### Exercise: Which lines ran?

Write `lines_run(func, *args)`: call `func(*args)` under `sys.settrace`, and return the set of line numbers **within `func`'s own body** that ran, counted from the `def` line (so the first line of the body is 1). Trace only `func`'s own frames (compare `frame.f_code` with `func.__code__`), restore the previous trace function afterwards (`sys.gettrace()`), even if `func` raises, and let the exception propagate.

In [ ]:
import sys

def lines_run(func, *args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lines():\n    \"tracing one function\"\n    import sys\n    def sign(n):\n        if n > 0:\n            return \"positive\"\n        if n < 0:\n            return \"negative\"\n        return \"zero\"\n    assert lines_run(sign, 5) == {1, 2} and lines_run(sign, -1) == {1, 3, 4} and lines_run(sign, 0) == {1, 3, 5}\n    def boom():\n        x = 1\n        raise ValueError(x)\n    try:\n        lines_run(boom)\n        assert False\n    except ValueError:\n        pass\n    assert sys.gettrace() is None\n    def helper():\n        return 1\n    def outer():\n        return helper() + 1\n    assert lines_run(outer) == {1}")

### Exercise: A coverage report

Write `coverage(source, run)`: compile `source` with the file name `"target.py"`, execute it into a new namespace, then call `run(ns)` (a function that exercises the code, as tests would) under a tracer that records the lines run in `"target.py"` (the module's own top-level lines count, as they run when it's executed, so trace that too). Executable lines are the `lineno` of every `ast.stmt` node. Return `(percent, missing)`: the percentage of executable lines run, rounded to a whole number, and the missing lines as a string of ranges, such as `"4-5, 9"` (empty if none). Also write `ranges(numbers)` for that string.

In [ ]:
import ast, sys

def ranges(numbers):
    ___

def coverage(source, run):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_report():\n    \"lines, percentages and ranges\"\n    assert ranges([4, 5, 9]) == \"4-5, 9\" and ranges([]) == \"\" and ranges([1, 2, 3, 7, 8, 10]) == \"1-3, 7-8, 10\"\n    src = \"def grade(score):\\n    if score >= 70:\\n        return 'distinction'\\n    if score >= 40:\\n        return 'pass'\\n    return 'fail'\\n\\nLIMIT = 100\\n\"\n    assert coverage(src, lambda ns: ns[\"grade\"](85)) == (57, \"4-6\")\n    assert coverage(src, lambda ns: [ns[\"grade\"](s) for s in (85, 50, 10)]) == (100, \"\")\n    assert coverage(src, lambda ns: None) == (29, \"2-6\")\n    import sys\n    assert sys.gettrace() is None")

### Exercise: Branches never taken

Write `arcs_run(func, calls)`: run `func(*args)` for each tuple of arguments in `calls`, recording the **arcs** `(from, to)` between consecutive lines in each call (line numbers counted from the `def` line, with the body starting at 1; the first arc of each call starts from 0). Then `untaken(func, calls, possible)`: given the set of `possible` arcs, return the sorted list of those that no call took.

In [ ]:
import sys

def arcs_run(func, calls):
    ___

def untaken(func, calls, possible):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_arcs():\n    \"branch coverage\"\n    def clamp(x, low, high):\n        if x < low:\n            x = low\n        if x > high:\n            x = high\n        return x\n    assert arcs_run(clamp, [(-5, 0, 10)]) == {(0, 1), (1, 2), (2, 3), (3, 5)}\n    possible = {(0, 1), (1, 2), (1, 3), (2, 3), (3, 4), (3, 5), (4, 5)}\n    assert untaken(clamp, [(-5, 0, 10)], possible) == [(1, 3), (3, 4), (4, 5)]\n    assert untaken(clamp, [(-5, 0, 10), (5, 0, 10), (50, 0, 10)], possible) == []\n    import sys\n    assert sys.gettrace() is None")

### Exercise: Making mutants

Write `mutants(source)`: a generator yielding `(label, mutated_source)` for every single-point mutation of these kinds, in the order the nodes appear in `ast.walk`: each comparison operator in `SWAPS` is replaced by its partner (`<` and `<=`, `>` and `>=`, `==` and `!=`), and each `+` or `-` binary operator becomes the other. The label is `f"line {lineno}: {old} -> {new}"`, using the operator class names (such as `Lt -> LtE`). Mutate the **first** operator of each comparison only. Build each mutant from a fresh parse, and produce source with `ast.unparse`.

In [ ]:
import ast

SWAPS = {ast.Lt: ast.LtE, ast.LtE: ast.Lt, ast.Gt: ast.GtE, ast.GtE: ast.Gt,
         ast.Eq: ast.NotEq, ast.NotEq: ast.Eq, ast.Add: ast.Sub, ast.Sub: ast.Add}

def mutants(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_mutants():\n    \"single-point mutants\"\n    src = \"def f(a, b):\\n    if a < b:\\n        return a + b\\n    return a == b\\n\"\n    got = list(mutants(src))\n    labels = [label for label, _ in got]\n    assert sorted(labels) == [\"line 2: Lt -> LtE\", \"line 3: Add -> Sub\", \"line 4: Eq -> NotEq\"], labels\n    by_label = dict(got)\n    assert \"a <= b\" in by_label[\"line 2: Lt -> LtE\"] and \"a - b\" in by_label[\"line 3: Add -> Sub\"]\n    assert \"a + b\" in by_label[\"line 2: Lt -> LtE\"] and \"a < b\" in by_label[\"line 4: Eq -> NotEq\"]\n    assert list(mutants(\"x = 1\\n\")) == [] and len(list(mutants(\"y = 1 * 2 - 3 + 4\\n\"))) == 2")

### Exercise: A mutation score

Using `mutants` from the previous exercise (given in the starter), write `mutation_score(source, name, tests)`: for each mutant, execute it into a new namespace and call `tests(ns[name])`; a mutant is **killed** if `tests` raises any exception (a crash counts as a kill). Return `(killed, total, survivors)`, where `survivors` is the list of surviving labels.

In [ ]:
import ast

SWAPS = {ast.Lt: ast.LtE, ast.LtE: ast.Lt, ast.Gt: ast.GtE, ast.GtE: ast.Gt,
         ast.Eq: ast.NotEq, ast.NotEq: ast.Eq, ast.Add: ast.Sub, ast.Sub: ast.Add}

def mutants(source):
    tree = ast.parse(source)
    count = len(list(ast.walk(tree)))
    for index in range(count):
        tree = ast.parse(source)
        node = list(ast.walk(tree))[index]
        if isinstance(node, ast.Compare) and type(node.ops[0]) in SWAPS:
            old = type(node.ops[0])
            node.ops[0] = SWAPS[old]()
        elif isinstance(node, ast.BinOp) and type(node.op) in SWAPS:
            old = type(node.op)
            node.op = SWAPS[old]()
        else:
            continue
        yield f"line {node.lineno}: {old.__name__} -> {SWAPS[old].__name__}", ast.unparse(tree)

def mutation_score(source, name, tests):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_score():\n    \"killed and surviving mutants\"\n    src = \"def grade(score):\\n    if score >= 70:\\n        return 'distinction'\\n    if score >= 40:\\n        return 'pass'\\n    return 'fail'\\n\"\n    def weak(grade):\n        assert grade(85) == \"distinction\" and grade(50) == \"pass\" and grade(10) == \"fail\"\n    def strong(grade):\n        weak(grade)\n        assert grade(70) == \"distinction\" and grade(40) == \"pass\"\n    assert mutation_score(src, \"grade\", weak) == (0, 2, [\"line 2: GtE -> Gt\", \"line 4: GtE -> Gt\"])\n    assert mutation_score(src, \"grade\", strong) == (2, 2, [])\n    def crashes(f):\n        f(None)\n    assert mutation_score(\"def g(x):\\n    return x + 1\\n\", \"g\", crashes)[0] == 1")

### Exercise: Killing the survivors

`shipping.py` has `shipping_cost(total, express)`, and its existing tests have 100% line coverage but let every mutant below survive. Write pytest tests that pass on the correct code and kill **all** the mutants.

In [ ]:
# Run this cell first: it creates the files used below.
with open('shipping.py', 'w', encoding='utf-8') as f:
    f.write('def shipping_cost(total, express):\n    if total >= 50 and not express:\n        return 0\n    cost = 4 if total < 20 else 3\n    if express:\n        cost = cost + 5\n    return cost\n')

In [ ]:
from shipping import shipping_cost

def test_free():
    assert shipping_cost(80, False) == 0

def test_small_order():
    assert shipping_cost(10, False) == 4

def test_express():
    assert shipping_cost(30, True) == 8

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"shipping.py\").read()\nBROKEN = {\n    \"line 2: GtE -> Gt\": GOOD.replace(\"total >= 50\", \"total > 50\"),\n    \"line 2: and -> or\": GOOD.replace(\"total >= 50 and not express\", \"total >= 50 or not express\"),\n    \"line 4: Lt -> LtE\": GOOD.replace(\"total < 20\", \"total <= 20\"),\n    \"line 4: 3 -> 4\": GOOD.replace(\"else 3\", \"else 4\"),\n    \"line 6: Add -> Sub\": GOOD.replace(\"cost + 5\", \"cost - 5\"),\n    \"line 2: express ignored\": GOOD.replace(\" and not express\", \"\"),\n}\ndef test_kills():\n    \"your tests pass on the correct code and kill all six mutants\"\n    results = against(source, \"shipping\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"shipping.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    survived = [k for k in BROKEN if not results[k]]\n    assert not survived, f\"survivors: {survived}\"")

### Bug hunt: Covered, but not checked

This test runs every line of `discount` (100% coverage) but checks nothing, so it passes on the broken version too. Turn it into a real test, with expected values worked out by hand, so the broken version is caught.

In [ ]:
# Run this cell first: it creates the files used below.
with open('discount.py', 'w', encoding='utf-8') as f:
    f.write('def discount(price, member):\n    if member:\n        return round(price * 0.9, 2)\n    return price\n')

In [ ]:
from discount import discount

def test_discount():
    discount(100, True)
    discount(100, False)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"the broken version is caught\"\n    good = open(\"discount.py\").read()\n    broken = good.replace(\"price * 0.9\", \"price * 0.8\")\n    results = against(source, \"discount\", {\"correct\": good, \"20% off\": broken, \"members pay full\": good.replace(\"return round(price * 0.9, 2)\", \"return price\")})\n    write_file(\"discount.py\", good)\n    assert results[\"correct\"] == [] and results[\"20% off\"] and results[\"members pay full\"], results")

### Bug hunt: A tracer that sees nothing

The global trace function records lines but returns `None`, so Python never gives it the frame's `"line"` events, and this prints `set()`. Return the tracer, so the frame is traced, and this prints `{1, 2}`.

In [ ]:
import sys

def double(x):
    y = x * 2
    return y

seen = set()
first = double.__code__.co_firstlineno

def tracer(frame, event, arg):
    if frame.f_code is double.__code__ and event == "line":
        seen.add(frame.f_lineno - first)

sys.settrace(tracer)
double(4)
sys.settrace(None)
print(seen)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a local tracer\"\n    expect_output(\"{1, 2}\")")

### Bug hunt: More than everything

The coverage counts every executed line, including ones that aren't statements of the module (the trace also saw line 9, in a different function), so this prints `125%`. Only count executed lines that are executable, so it prints `100%`.

In [ ]:
executable = {1, 2, 3, 5}
executed = {1, 2, 3, 5, 9}

percent = round(100 * len(executed) / len(executable))
print(f"{percent}%")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"lines that count\"\n    expect_output(\"100%\")")

### Puzzle: Tracing one function's lines

Arrange the lines of a tracer that records the lines run inside `target`, and nothing else. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: 100% line coverage means every branch was taken
q = ___   # the mutation score of a suite that kills 15 of 20 mutants, as a percentage
r = ___   # "killed" or "survived": a mutant for which every test still passes
s = ___   # True or False: an equivalent mutant can be killed by a better test
t = ___   # what sys.settrace's global tracer must return for a frame to get "line" events: "a tracer" or "True"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q == 75 and r == \"survived\", \"p, q and r\"\n    assert s is False and t == \"a tracer\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a mutation tester with more operators

Write `mutate(source)`: a generator of `(label, mutated_source)` with **four** kinds of single-point mutation, found in `ast.walk` order: comparisons (first operator only, as in `SWAPS_CMP`), arithmetic (`+` and `-`, `*` and `/`), boolean operators (`and` and `or`), and integer constants (every `int` constant `n`, but not `True` or `False`, becomes `n + 1`). Labels are `f"line {lineno}: {description}"`, where the description is `"Lt -> LtE"` style for operators and `"5 -> 6"` for constants. Then `report(source, name, tests)`: run `tests(ns[name])` against every mutant (an exception kills it) and return a dict with `"score"` (killed over total as a whole-number percentage, 100 if there are no mutants), `"killed"` and `"survivors"` (a list of labels).

In [ ]:
import ast

SWAPS_CMP = {ast.Lt: ast.LtE, ast.LtE: ast.Lt, ast.Gt: ast.GtE, ast.GtE: ast.Gt, ast.Eq: ast.NotEq, ast.NotEq: ast.Eq}
SWAPS_OP = {ast.Add: ast.Sub, ast.Sub: ast.Add, ast.Mult: ast.Div, ast.Div: ast.Mult}
SWAPS_BOOL = {ast.And: ast.Or, ast.Or: ast.And}

def mutate(source):
    ___

def report(source, name, tests):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_mutation_tester():\n    \"four operators and a report\"\n    src = \"def price(qty, member):\\n    if qty > 10 and member:\\n        return qty * 9\\n    return qty * 10\\n\"\n    labels = sorted(label for label, _ in mutate(src))\n    assert labels == sorted([\"line 2: Gt -> GtE\", \"line 2: And -> Or\", \"line 2: 10 -> 11\", \"line 3: Mult -> Div\", \"line 3: 9 -> 10\", \"line 4: Mult -> Div\", \"line 4: 10 -> 11\"]), labels\n    assert all(\"True\" not in m for _, m in mutate(\"x = True\\n\")) and list(mutate(\"x = True\\n\")) == []\n    def weak(price):\n        assert price(20, True) == 180 and price(5, False) == 50\n    r = report(src, \"price\", weak)\n    assert r[\"killed\"] == 4 and r[\"score\"] == 57 and sorted(r[\"survivors\"]) == [\"line 2: 10 -> 11\", \"line 2: And -> Or\", \"line 2: Gt -> GtE\"], r\n    def strong(price):\n        weak(price)\n        assert price(10, True) == 100 and price(11, True) == 99 and price(20, False) == 200\n    assert report(src, \"price\", strong) == {\"score\": 100, \"killed\": 7, \"survivors\": []}\n    assert report(\"x = 'a'\\n\", \"x\", lambda x: None)[\"score\"] == 100")

## Warm-up

Every line ran, so this test has **100% coverage**, and it would pass if `grade` returned the wrong word every time, because it never checks anything. Coverage measures what the tests **run**, not what they **check**. Keep that in mind all day.

## Line coverage

Python can call a function of yours on every event while code runs: `sys.settrace(tracer)` installs a **global** trace function, called with `(frame, event, arg)` whenever a new function is entered (`event == "call"`). Whatever it **returns** becomes the **local** trace function for that frame, which then receives `"line"` events (a new line is about to run), `"return"` and `"exception"`. Returning `None` stops tracing that frame. `frame.f_code` identifies the function and `frame.f_lineno` is the current line. This is how debuggers and coverage tools work.

To measure line coverage of a module:

1. Find its **executable lines**: the line numbers of its statements. `ast.walk(ast.parse(source))` visits every node; those that are `ast.stmt` have a `lineno`. (Coverage tools also skip docstrings and lines like `else:`, which aren't statements.)
2. Run the tests under a tracer that records `frame.f_lineno` for each `"line"` event whose `frame.f_code.co_filename` is the module's file.
3. Coverage is executed lines over executable lines, and the **missing** lines (executable but never run) are the useful part of the report: they show exactly which code no test touches.

Line 1 (`def grade`) counts as executed because the module ran it when it defined the function. The missing line is `return "pass"`: no test had a score from 40 to 69.

## Branch coverage, and what coverage can't tell you

Line coverage misses things. `if x: y = 1` on one line, or an `if` with no `else`, can be fully "covered" by tests that only ever take one path. **Branch coverage** records **arcs**, pairs `(from line, to line)` of consecutive lines in the same frame, and checks that every possible jump was taken: the `if` both true and false, the loop both entered and skipped.

And then remember what coverage **can't** tell you:

- **Covered isn't checked.** A line can run in a test that asserts nothing about it (the warm-up).
- **It can't see missing code.** If the function forgot to handle negative numbers, there's no line for the missing check, and coverage is still 100%.
- **It doesn't count inputs.** `x / y` is covered by one test with `y = 2`; the bug with `y = 0` is untouched.

So **low coverage is a reliable warning** (that code is untested) and **high coverage is weak evidence** (the code ran). Use the missing-lines report to find gaps, not the percentage as a target: a team told to reach 90% will write tests that run code without checking it.

One call ran every line, so line coverage is 100%, yet the arcs show the path where `x` is already in range was never taken, and neither was the path where `x > high`.

## Mutation testing

Mutation testing measures what coverage can't: whether the tests **detect** changes.

1. Make a **mutant**: a copy of the code with one small change of the kind people really get wrong. Common **mutation operators**: replace `<` with `<=` (and the other comparisons), `+` with `-`, `and` with `or`; change a constant (`0` to `1`); negate a condition; replace a returned value with `None`.
2. Run the whole test suite against the mutant. If some test fails, the mutant is **killed**. If every test passes, it **survived**: a bug of that kind would go unnoticed.
3. The **mutation score** is killed over total. Every survivor is a specific, actionable hint: "nothing tests `score == 70`" is far more useful than "line 2 is covered".

The `ast` module makes mutants safely: parse the source, change one node with an `ast.NodeTransformer` (or by counting nodes and changing the *n*th one), then `ast.unparse` the tree back to source.

Some mutants are **equivalent**: the change doesn't alter behaviour at all (`x >= 0` versus `x > -1` for integers, or a mutation in code whose result is never used). No test can kill them. Mark them and move on, and treat a survivor as equivalent only after trying to write the test that would kill it.

Mutation testing is slow (every mutant runs the whole suite), so real tools run only the tests that cover the mutated line, and you run it occasionally, on the modules that matter most.

The weak tests have 100% line coverage and kill **neither** mutant; testing the boundaries kills both.

## The real tools

On your own computer, use **coverage.py** (with pytest, through **pytest-cov**) for coverage, and **mutmut** for mutation testing:

```bash
pip install pytest-cov mutmut
```

```bash
pytest --cov=myproject --cov-branch --cov-report=term-missing
```

```bash
mutmut run
```

`--cov-report=term-missing` prints the missing lines for each file, exactly the report you'll build today; `--cov-branch` adds branch coverage; `--cov-fail-under=80` makes the run fail below a threshold (useful in CI, as a floor rather than a goal). `mutmut run` mutates your code and runs the tests against each mutant; `mutmut results` lists the survivors and `mutmut show` prints a survivor's change. A `# pragma: no cover` comment excludes a line from coverage, for code that truly can't be tested, such as an `if __name__ == "__main__":` block.

## Practice

## Quiz

## Daily challenge

## Before you go

Run today's mutation tester on a function from an earlier week, with the tests you wrote then, and look at the survivors. Tomorrow: **test architecture**, organising a whole suite: the test pyramid, unit and integration tests, where test doubles belong, and keeping a large suite fast and trustworthy.